# 4. CNN

Module importing

In [2]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, "..")

Library importing

In [3]:
import numpy as np
import torch
from src.data import load_cache
from src.splits import SPLITS
from src.features import build_feature_table
from src.cnn import train_cnn, predict_cnn
from src.evaluate import score

cells = load_cache()
tables = {name: build_feature_table(cells, ids) for name, ids in SPLITS.items()}
results = {}

print({k: len(v) for k, v in tables.items()})

{'train': 41, 'primary_test': 42, 'secondary_test': 40}


CNN training

In [4]:
print("device:", "cuda" if torch.cuda.is_available() else "cpu")

train_ids = list(tables["train"].index)
rng = np.random.default_rng(0)
val_ids = list(rng.choice(train_ids, size=8, replace=False))
fit_ids = [c for c in train_ids if c not in val_ids]

curves = lambda ids: [cells[i]["qdlin"] for i in ids]
lives = lambda ids: tables["train"].loc[ids, "cycle_life"].values

cnns = [train_cnn(curves(fit_ids), lives(fit_ids), curves(val_ids), lives(val_ids), seed=s)
        for s in range(5)]

for split in ["primary_test", "secondary_test"]:
    ids, truth = list(tables[split].index), tables[split]["cycle_life"]
    per_seed = np.array([predict_cnn(m, curves(ids)) for m in cnns])
    singles = [score(truth, p)["RMSE (cycles)"] for p in per_seed]
    print(split, "single seeds:", singles, "| ensemble:", score(truth, per_seed.mean(0)))

results["1D CNN (5-seed ensemble)"] = {
    s: (t["cycle_life"], np.mean([predict_cnn(m, curves(list(t.index))) for m in cnns], axis=0))
    for s, t in tables.items()
}

device: cuda
primary_test single seeds: [321.2, 304.7, 291.1, 298.8, 393.6] | ensemble: {'RMSE (cycles)': 317.6, 'MAPE (%)': 14.8}
secondary_test single seeds: [369.3, 366.7, 299.7, 344.9, 507.9] | ensemble: {'RMSE (cycles)': 372.9, 'MAPE (%)': 21.0}


In [5]:
import joblib
from src.config import MODELS_DIR
from src.cnn import save_cnn

for s, m in enumerate(cnns):
    save_cnn(m, MODELS_DIR / f"cnn_seed{s}.pt")

spread = {}
for split in ["primary_test", "secondary_test"]:
    ids, truth = list(tables[split].index), tables[split]["cycle_life"]
    spread[split] = [score(truth, predict_cnn(m, curves(ids)))["RMSE (cycles)"] for m in cnns]
joblib.dump(results, MODELS_DIR / "results_cnn.joblib")
joblib.dump(spread, MODELS_DIR / "cnn_seed_spread.joblib")
print(spread)

{'primary_test': [321.2, 304.7, 291.1, 298.8, 393.6], 'secondary_test': [369.3, 366.7, 299.7, 344.9, 507.9]}
